In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import fabio

from scipy.stats import binned_statistic_2d

In [ ]:
dir_coupled = r"C:\Users\j.bantol\Documents\Data\RSM\2026-06-29_gao_sto01.3\1_EIGERfull-2Dsnapshot_RSM-STO026s_34.9-37deg_0.01deg_3s_FrameFiles\frames"
dir_rocking = r"C:\Users\j.bantol\Documents\Data\RSM\2026-06-29_gao_sto01.3\2_EIGERfull-2Dsnapshot_rocking-STO026s_34.5-36deg_0.01deg_0.1s_FrameFiles\frames"

In [ ]:
%matplotlib widget
plt.close("all")

frames = sorted([f for f in os.listdir(dir_coupled) if f.endswith((".gfrm"))])
print(f"Found {len(frames)} frames")

# plot reference frame
ref_obj = fabio.open(os.path.join(dir_coupled, frames[40]))
ref_data = ref_obj.data
#ref_data = np.rot90(ref_obj.data, k=1)
print("pixel size:", ref_data.shape)

fig, ax = plt.subplots(figsize=(8, 5))
img = ax.imshow(ref_data, vmax=np.percentile(ref_data, 99.5), origin="lower", cmap="viridis")
plt.colorbar(img, ax=ax, label="Intensity / counts")


ax.set_title(f"Reference frame", fontsize="12")
ax.set_xlabel(f"pixel", fontsize="11")
ax.set_ylabel(f"pixel", fontsize="11")
plt.show()

In [ ]:
# instrument parameters
wavelength = 0.71076      # Å, Mo Kα
pixel_size = 0.075        # mm (75 µm)
detector_distance = 310   # mm
center_col = 493.98       # px, beam center column
center_row = 226.5        # px, beam center row    

In [ ]:
# for each frame, convert center pixel to 2theta
def pixel_to_2theta(pixel, center_pixel, two_theta_center, detector_distance, pixel_size): 
    offset_mm = (pixel - center_pixel) * pixel_size
    delta_2theta = np.rad2deg(np.arctan(offset_mm / detector_distance))    # opposite is offset_mm
    
    return two_theta_center + delta_2theta

# convert omega, 2theta to Qy, Qz
def angles_to_Q(omega, two_theta, wavelength, chi=18.4349):     # chi=tan-1(1/6 / 1/2) for angle between (026) and (001)
    k = 2 * np.pi / wavelength
    alpha_i = np.deg2rad(omega)
    alpha_f = np.deg2rad(two_theta - omega)
    chi_r   = np.deg2rad(chi)

    # Q in diffractometer frame (before tilt)
    Q_perp = k * (np.sin(alpha_i) + np.sin(alpha_f))  # along diffractometer z
    Q_par  = k * (np.cos(alpha_i) - np.cos(alpha_f))  # along diffractometer y

    # rotate by chi to get into crystal frame (026s geometry, phi=-90)
    Qy =  Q_perp * np.sin(chi_r) + Q_par * np.cos(chi_r)
    Qz =  Q_perp * np.cos(chi_r) - Q_par * np.sin(chi_r)

    return Qy, Qz

In [ ]:
# for one frame

## scan parameters
### coupled omega-2theta scan
omega = 34.9
count_time_c = 3.0
two_theta_center = 2 * omega
chi=18.4349

In [ ]:
frames = sorted([f for f in os.listdir(dir_coupled) if f.endswith((".gfrm"))])
obj = fabio.open(os.path.join(dir_coupled, frames[0]))
data = obj.data.astype(float) / count_time_c

nrows = data.shape[0]
ncols = data.shape[1]

Qy_frame = []
Qz_frame = []
I_frame = []

for col in range(ncols):
    for row in range(nrows):
        intensity = data[row, col]
        if intensity <= 0:
            continue
            
        # cols to 2theta to Qy and Qz via chi rotation
        tt_pixel = pixel_to_2theta(col, center_col, two_theta_center, detector_distance, pixel_size)
        Qy_pix, Qz_pix = angles_to_Q(omega, tt_pixel, wavelength)
        
        # rows to rows with offset
        offset_row = (row - center_row) * pixel_size
        delta_2theta_row = np.rad2deg(np.arctan(offset_row / detector_distance))
        delta_Qy = (2 * np.pi / wavelength) * np.deg2rad(delta_2theta_row)
        
        Qy_frame.append(Qy_pix + delta_Qy)
        Qz_frame.append(Qz_pix)
        I_frame.append(intensity)

Qy_frame = np.array(Qy_frame)
Qz_frame = np.array(Qz_frame)
I_frame  = np.array(I_frame)

In [ ]:
I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(Qy_frame, Qz_frame, I_frame, statistic='mean', bins=[200, 200],
                                                    range=[[Qy_frame.min(), Qy_frame.max()], [Qz_frame.min(), Qz_frame.max()]])
I_grid = np.nan_to_num(I_grid, nan=0.0)
Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                   cmap='hot',
                   vmin=np.nanpercentile(np.log10(I_grid[I_grid>0]+1), 50),
                   vmax=np.nanpercentile(np.log10(I_grid+1), 99.99))
plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')
ax.set_xlabel("Qy / Å⁻¹")
ax.set_ylabel("Qz / Å⁻¹")
ax.set_title(f"Single frame RSM — ω={omega:.3f}°")
plt.show()

In [ ]:
obj = fabio.open(os.path.join(dir_coupled, frames[0]))
data = obj.data

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(data.sum(axis=0))  # vs col
axes[0].set_xlabel("col")
axes[0].set_title("horizontal profile")

axes[1].plot(data.sum(axis=1))  # vs row
axes[1].set_xlabel("row")
axes[1].set_title("vertical profile")

plt.tight_layout()
plt.show()

In [ ]:
# for multiple frames

## scan parameters
### coupled omega-2theta scan
omega_start_c = 34.9
omega_step_c = 0.01
count_time_c = 3.0

### rocking scan
omega_start_r = 34.5
omega_step_r = 0.01
count_time_r = 0.1
two_theta_fixed_r = 70.1144  # fixed, 2theta = 2 * omega

In [ ]:
all_Qy = []
all_Qz = []
all_I = []
all_sctype = []

def process_data(directory, omega_start, omega_step, count_time, scan_type="coupled", two_theta_fixed=None):
    frames = sorted([f for f in os.listdir(directory) if f.endswith((".gfrm"))])
    print(f"For {scan_type} scan: {len(frames)} frames")
    
    for i, f in enumerate(frames):
        obj = fabio.open(os.path.join(directory, f))
        data = obj.data.astype(float) / count_time  # normalize to cps
        #data = np.rot90(orig_data, k=1)
        nrows, ncols = data.shape
    
        omega = omega_start + i * omega_step
    
        if scan_type == "coupled":
            two_theta_center = 2 * omega
        else:
            two_theta_center = two_theta_fixed
    
        # cols -> 2theta -> Qy/Qz via chi rotation, vectorized over the whole column axis
        cols = np.arange(ncols)
        tt_pixel = pixel_to_2theta(cols, center_col, two_theta_center, detector_distance, pixel_size)
        Qy_col, Qz_col = angles_to_Q(omega, tt_pixel, wavelength)          # shape (ncols,)
    
        # rows -> offset -> delta_Qy, vectorized over the whole row axis
        rows = np.arange(nrows)
        offset_row = (rows - center_row) * pixel_size
        delta_Qy = (2 * np.pi / wavelength) * np.arctan(offset_row / detector_distance)  # shape (nrows,)
    
        Qy_grid = Qy_col[None, :] + delta_Qy[:, None]       # shape (nrows, ncols), matches data
        Qz_grid = np.broadcast_to(Qz_col[None, :], data.shape)
    
        mask = data > 0
        all_Qy.append(Qy_grid[mask])
        all_Qz.append(Qz_grid[mask])
        all_I.append(data[mask])
        all_sctype.append(np.full(mask.sum(), scan_type))
    
    global Qy, Qz, I, sctype
    Qy = np.concatenate(all_Qy)
    Qz = np.concatenate(all_Qz)
    I = np.concatenate(all_I)
    sctype = np.concatenate(all_sctype)


In [ ]:
# process a scan
process_data(dir_coupled, omega_start_c, omega_step_c, count_time_c, scan_type="coupled")
process_data(dir_rocking, omega_start_r, omega_step_r, count_time_r, scan_type="rocking", two_theta_fixed=two_theta_fixed_r)


In [ ]:
print(f"Total points: {len(Qy)}")
print(f"Qy range: {Qy.min():.4f} to {Qy.max():.4f} Å⁻¹")
print(f"Qz range: {Qz.min():.4f} to {Qz.max():.4f} Å⁻¹")

In [ ]:
grids = {}
for label in ['coupled', 'rocking']:
    mask = sctype == label
    Qy_s, Qz_s, I_s = Qy[mask], Qz[mask], I[mask]

    I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(
        Qy_s, Qz_s, I_s, statistic='mean', bins=[200, 200],
        range=[[Qy_s.min(), Qy_s.max()], [Qz_s.min(), Qz_s.max()]]
    )
    I_grid = np.nan_to_num(I_grid, nan=0.0)
    Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
    Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

    grids[label] = (I_grid, Qy_centers, Qz_centers)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, label in zip(axes, ['coupled', 'rocking']):
    I_grid, Qy_centers, Qz_centers = grids[label]
    im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                        cmap='hot',
                        vmin=np.nanpercentile(np.log10(I_grid[I_grid > 0] + 1), 50),
                        vmax=np.nanpercentile(np.log10(I_grid + 1), 99.99))
    plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')
    ax.set_xlabel("Qy / Å⁻¹")
    ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(f"{label} scan RSM ({(sctype == label).sum()} pts)")

plt.tight_layout()
plt.show()


In [ ]:
for label, (I_grid, _, _) in grids.items():
    print(f"{label}: valid intensity sum = {np.nansum(I_grid):.1f}, NaN count = {np.sum(np.isnan(I_grid))}")
print("Qy sample:", Qy[:5])
print("Qz sample:", Qz[:5])
print("I sample:", I[:5])


In [ ]:
# grid and plot
I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(Qy, Qz, I, statistic="mean", bins=[600, 600], range=[[Qy.min(), Qy.max()], [Qz.min(), Qz.max()]])

#Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
#Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

I_grid = np.nan_to_num(I_grid, nan=0.0)  # replace NaN with 0 for plotting

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, label, color in zip(axes, ['rocking', 'coupled'], ['C0', 'C1']):
    mask = sctype == label
    ax.scatter(Qy[mask], Qz[mask], c=np.log10(I[mask]+1), s=0.1,
               cmap='viridis')
    ax.set_xlabel("Qy / Å⁻¹")
    ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(f"{label}: {mask.sum()} points")

plt.tight_layout()
plt.show()
